# 10 — Cross-Sectional XGBoost Ranking Study

This notebook compares three ways to score stocks within each trading date:

- regression on future market-relative return;
- classification of future top-quintile membership;
- learning to rank with `XGBRanker` and ordinal relevance grades.

It is an exploration notebook, not part of the reusable production training harness. It fits on train and evaluates validation only. Keep the committed notebook in this generic state; export materially changed studies to `notebooks/exports/` as HTML before restoring the base notebook.


In [ ]:
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml
from plotly.subplots import make_subplots
from xgboost import XGBClassifier, XGBRanker, XGBRegressor

from swingtrader import indicators
from swingtrader.core.paths import find_repo_root
from swingtrader.data import features
from swingtrader.data.bronze.loaders import load_bronze_daily_prices
from swingtrader.data.bronze.queries import load_available_tickers
from swingtrader.data.db import resolve_database_engine
from swingtrader.modeling.datasets import (
    CROSS_SECTIONAL_RETURN_PRIMARY_TASK,
    CROSS_SECTIONAL_RETURN_TARGET_SET,
    TemporalDatasetSpec,
    UniverseSpec,
    build_temporal_dataset,
    to_tabular_dataset,
)
from swingtrader.modeling.experiments import FixedTemporalSplitter, TemporalSplitSpec

RANDOM_SEED = 23
TOP_K = 10
HORIZON = 5
TOP_QUANTILE_THRESHOLD = 0.80
PROVIDER = "yfinance"

TRAIN_START = date(2010, 1, 1)
TRAIN_END = date(2020, 12, 31)
VALIDATION_START = date(2021, 1, 1)
VALIDATION_END = date(2023, 12, 31)
TEST_START = date(2024, 1, 1)
TEST_END = date(2025, 12, 31)

repo_root = find_repo_root()
database_url = f"sqlite+pysqlite:///{(repo_root / 'data' / 'swingtrader.sqlite').as_posix()}"
ENGINE = resolve_database_engine(database_url=database_url)

## Resolve the exploratory universe

The cross-sectional targets need many stocks on each date. This base notebook reads the current Large Cap and Mid Cap configuration files. Change the two paths or replace `TICKERS` with an explicit tuple for a narrower study.


In [ ]:
def configured_tickers(path: Path) -> tuple[str, ...]:
    config = yaml.safe_load(path.read_text(encoding="utf-8"))
    return tuple(item["ticker"] for item in config["symbols"])

universe_directory = repo_root / "src" / "swingtrader" / "configs" / "universes"
TICKERS = tuple(
    dict.fromkeys(
        configured_tickers(universe_directory / "se_large_cap.yml")
        + configured_tickers(universe_directory / "se_mid_cap.yml")
    )
)

available_tickers = load_available_tickers(
    engine=ENGINE,
    provider=PROVIDER,
    start_date=TRAIN_START,
    end_date=TEST_END,
)
TICKERS = tuple(ticker for ticker in TICKERS if ticker in set(available_tickers))

len(TICKERS), TICKERS[:5]

## Build the cross-sectional temporal dataset

The existing dataset and split contracts remain the source of truth. The selected supervised task is the continuous five-session percentile target, while the notebook also reads the relative-return and relevance-grade columns generated by the same target set.


In [ ]:
RESEARCH_FEATURE_SET = features.FeatureSetSpec(
    name="ohlcv_v1_candidates",
    version="1",
    blocks=(
        features.FeatureBlockSpec(
            name="returns",
            builder=features.add_return_features,
            parameters={"horizons": (1, 5, 10, 20)},
            output_columns=(
                # "return_1d",
                # "return_5d",
                "return_10d",
                "return_20d",
            ),
            required_columns=frozenset({"adjusted_close"}),
        ),
        features.FeatureBlockSpec(
            name="cross_sectional",
            builder=features.add_cross_sectional_features,
            parameters={
                "return_horizons": (1, 5, 10, 20),
                "market_return_horizon": 1,
                "minimum_cross_section_size": 2,
            },
            output_columns=(
                "return_1d_cross_sectional_percentile",
                "return_5d_cross_sectional_percentile",
                "return_10d_cross_sectional_percentile",
                "return_20d_cross_sectional_percentile",
                "market_breadth_positive_1d",
                "market_mean_return_1d",
                "market_median_return_1d",
            ),
            required_columns=frozenset({"adjusted_close"}),
        ),
        features.FeatureBlockSpec(
            name="trend",
            builder=features.add_trend_features,
            parameters={
                "ma_lengths": (10, 20, 50),
                "adx_length": 14,
                "rolling_fraction_lookback": 20,
                "vwap_length": 20,
                "vwap_bollinger_length": 20,
                "vwap_bollinger_num_std": 2.0,
            },
            output_columns=(
                "ema_fast_to_ema_mid",
                "ema_mid_to_ema_slow",
                "ema_mid_to_sma_mid",
                "close_to_ema_fast",
                "close_to_ema_mid",
                "close_to_ema_slow",
                # "close_over_ema_fast_fraction",
                # "close_over_ema_mid_fraction",
                # "close_over_ema_slow_fraction",
                # "adx",
                # "plus_di",
                # "minus_di",
                # "vwap_distance",
                # "vwap_distance_percent_b",
            ),
            required_columns=frozenset({"high", "low", "close", "volume", "adjusted_close"}),
            history_requirement=features.HistoryRequirement.EXPANDING,
        ),
        features.FeatureBlockSpec(
            name="momentum",
            builder=features.add_momentum_features,
            parameters={
                "ppo_lengths": (12, 26, 9),
                "ppo_percentile_min_history": 100,
                "rsi_length": 21,
                "rsi_bollinger_length": 20,
                "rsi_bollinger_num_std": 2.0,
                "stochastic_k_length": 14,
                "stochastic_k_smoothing": 3,
                "stochastic_d_length": 3,
                "mfi_length": 14,
                "mfi_bollinger_length": 20,
                "mfi_bollinger_num_std": 2.0,
                "squeeze_bb_length": 20,
                "squeeze_bb_mult": 2.0,
                "squeeze_kc_length": 20,
                "squeeze_kc_mult": 1.5,
                "squeeze_atr_length": 14,
            },
            output_columns=(
                "ppo",
                # "ppo_signal",
                # "ppo_histogram",
                # "ppo_percentile",
                "rsi",
                "rsi_percent_b",
                # "stochastic_k",
                # "stochastic_d",
                # "mfi",
                # "mfi_percent_b",
                # "squeeze_on",
                # "squeeze_off",
                # "squeeze_released",
                # "squeeze_width_ratio",
                # "squeeze_momentum_atr",
                # "squeeze_momentum_atr_change",
                # "squeeze_duration",
                # "squeeze_release_duration",
            ),
            required_columns=frozenset({"high", "low", "close", "adjusted_close", "volume"}),
            history_requirement=features.HistoryRequirement.EXPANDING,
        ),
        features.FeatureBlockSpec(
            name="volatility",
            builder=features.add_volatility_features,
            parameters={
                "adr_length": 20,
                "atr_length": 14,
                "bollinger_length": 20,
                "bollinger_num_std": 2.0,
            },
            output_columns=(
                # "adr_percent",
                "atr_percent",
                # "bollinger_bandwidth",
                "bollinger_percent_b",
            ),
            required_columns=frozenset({"high", "low", "close", "adjusted_close"}),
            history_requirement=features.HistoryRequirement.EXPANDING,
        ),
        features.FeatureBlockSpec(
            name="price_action",
            builder=features.add_price_action_features,
            parameters={
                "atr_length": 14,
                "range_percentile_length": 20,
                "breakout_length": 20,
                "rolling_candle_lookback": 14,
            },
            output_columns=(
                "candle_signed_body_fraction",
                # "candle_upper_wick_fraction",
                # "candle_lower_wick_fraction",
                # "candle_close_location",
                "candle_range_atr",
                "candle_gap_atr",
                # "range_percentile_20",
                "candle_inside_bar",
                "candle_outside_bar",
                "candle_engulfing_strength",
                "candle_lower_rejection_strength",
                "candle_upper_rejection_strength",
                "candle_consecutive_inside_bars",
                "candle_direction_run",
                "candle_direction_run_return",
                "candle_direction_run_body_atr",
                "candle_close_to_prior_high_atr_20",
                "candle_close_to_prior_low_atr_20",
                "candle_breakout_high_strength_20",
                "candle_breakout_low_strength_20",
                # "candle_failed_breakout_high_strength_20",
                # "candle_failed_breakout_low_strength_20",
                "rolling_bullish_candle_fraction",
            ),
            required_columns=frozenset({"open", "high", "low", "close", "adjusted_close"}),
            history_requirement=features.HistoryRequirement.EXPANDING,
        ),
        # features.FeatureBlockSpec(
        #     name="volume",
        #     builder=features.add_volume_features,
        #     parameters={
        #         "turnover_zscore_length": 252,
        #         "turnover_zscore_log": True,
        #     },
        #     output_columns=(
        #         "turnover_zscore",
        #     ),
        #     required_columns=frozenset({"close", "volume"}),
        # ),
        # features.FeatureBlockSpec(
        #     name="market_structure",
        #     builder=features.add_market_structure_features,
        #     parameters={
        #         "donchian_length": 20,
        #         "zigzag_deviation": 5.0,
        #         "zigzag_pivot_legs": 10,
        #         "zigzag_consistency_pivots": 4,
        #         "zigzag_dynamics_legs": 6,
        #         "zigzag_atr_length": 14,
        #     },
        #     output_columns=(
        #         "donchian_position",
        #         # "zigzag_last_direction",
        #         # "zigzag_last_swing_return",
        #         # "zigzag_last_swing_bars",
        #         # # "zigzag_swing_return_per_bar",
        #         # "zigzag_bars_since_pivot",
        #         # "zigzag_retracement",
        #         # "market_structure_high_change",
        #         # "market_structure_low_change",
        #         # # "market_structure_high_rate",
        #         # # "market_structure_low_rate",
        #         # # "market_structure_high_consistency",
        #         # # "market_structure_low_consistency",
        #         # # "market_structure_leg_balance",
        #         # # "market_structure_efficiency",
        #         # # "market_structure_close_to_prior_high_atr",
        #         # # "market_structure_close_to_prior_low_atr",
        #         # "market_structure_breakout_high_strength",
        #         # "market_structure_breakout_low_strength",
        #         # "market_structure_failed_breakout_high_strength",
        #         # "market_structure_failed_breakout_low_strength",
        #     ),
        #     required_columns=frozenset({"high", "low", "close"}),
        #     history_requirement=features.HistoryRequirement.PATH_DEPENDENT,
        # ),
    ),
)

In [ ]:
universe = UniverseSpec(
    name="stockholm_large_mid_cap_exploration",
    version="1",
    provider=PROVIDER,
    tickers=TICKERS,
)
dataset_spec = TemporalDatasetSpec(
    feature_set=RESEARCH_FEATURE_SET,
    target_set=CROSS_SECTIONAL_RETURN_TARGET_SET,
    task=CROSS_SECTIONAL_RETURN_PRIMARY_TASK,
    universe=universe,
    data_start=TRAIN_START,
    data_end=TEST_END,
)
split_spec = TemporalSplitSpec(
    name="cross_sectional_ranking_holdout",
    version="1",
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    validation_start=VALIDATION_START,
    validation_end=VALIDATION_END,
    test_start=TEST_START,
    test_end=TEST_END,
)

bundle = build_temporal_dataset(engine=ENGINE, spec=dataset_spec)
split_result = FixedTemporalSplitter(split_spec).assign(bundle)
bundle.manifest.to_manifest(), split_result.manifest.to_manifest()

## Select train and validation rows

The locked test is intentionally not read in this notebook. All three learned models use the same generated feature matrix and the same outer train/validation ranges.


In [ ]:
relative_return_column = f"market_relative_forward_return_{HORIZON}d"
percentile_column = f"forward_return_{HORIZON}d_cross_sectional_percentile"
relevance_column = f"forward_return_{HORIZON}d_relevance_grade"

tabular = to_tabular_dataset(bundle)
train_positions = split_result.indices("train")
validation_positions = split_result.indices("validation")

X_train = tabular.X.iloc[train_positions]
X_validation = tabular.X.iloc[validation_positions]
relative_return_train = bundle.targets[relative_return_column].iloc[train_positions]
relative_return_validation = bundle.targets[relative_return_column].iloc[validation_positions]
percentile_train = bundle.targets[percentile_column].iloc[train_positions]
percentile_validation = bundle.targets[percentile_column].iloc[validation_positions]
relevance_train = bundle.targets[relevance_column].iloc[train_positions]
relevance_validation = bundle.targets[relevance_column].iloc[validation_positions]
classification_train = percentile_train.ge(TOP_QUANTILE_THRESHOLD).astype("int8")

{
    "train_rows": len(X_train),
    "validation_rows": len(X_validation),
    "feature_count": len(X_train.columns),
    "train_dates": X_train.index.get_level_values("trading_date").nunique(),
    "validation_dates": X_validation.index.get_level_values("trading_date").nunique(),
    "top_quantile_prevalence": classification_train.mean(),
}


## Fit the three XGBoost variants

These are deliberately compact starting parameters for comparison, not a tuned search space. XGBoost handles the retained feature missing values directly.


In [ ]:
common_parameters = {
    "n_estimators": 200,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "min_child_weight": 20,
    "reg_lambda": 3.0,
    "tree_method": "hist",
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
}

regressor = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    **common_parameters,
)
classifier = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    **common_parameters,
)
ranker = XGBRanker(
    objective="rank:ndcg",
    eval_metric=f"ndcg@{TOP_K}",
    **common_parameters,
)


In [ ]:
regressor.fit(X_train, relative_return_train)
classifier.fit(X_train, classification_train)

`XGBRanker` requires rows belonging to the same query to be contiguous. Here one query is one provider and trading date.


In [ ]:
from swingtrader.modeling.training import prepare_xgboost_ranking_data

X_rank_train, relevance_rank_train, train_query_ids = prepare_xgboost_ranking_data(
    X_train,
    relevance_train,
)
ranker.fit(X_rank_train, relevance_rank_train, qid=train_query_ids)


## Score validation and compare ranking diagnostics

All outputs are treated as ranking scores. The random baseline uses the existing deterministic index-based score generator. Rank IC is Spearman correlation with continuous market-relative return; NDCG uses the ordinal relevance grades.


In [ ]:
from swingtrader.modeling.training import (
    deterministic_random_scores,
    evaluate_cross_sectional_scores,
)

validation_scores = {
    "random": deterministic_random_scores(X_validation.index, seed=RANDOM_SEED),
    "regression": pd.Series(
        regressor.predict(X_validation),
        index=X_validation.index,
        name="score",
    ),
    "classification": pd.Series(
        classifier.predict_proba(X_validation)[:, 1],
        index=X_validation.index,
        name="score",
    ),
}

X_rank_validation, _, _ = prepare_xgboost_ranking_data(
    X_validation,
    relevance_validation,
)
ranker_scores = pd.Series(
    ranker.predict(X_rank_validation),
    index=X_rank_validation.index,
    name="score",
).reindex(X_validation.index)
validation_scores["ranking"] = ranker_scores

df_scores = pd.DataFrame(validation_scores)

summaries = {}
daily_results = {}
for model_name, scores in df_scores.items():
    summaries[model_name], daily_results[model_name] = evaluate_cross_sectional_scores(
        scores,
        relevance_validation,
        relative_return_validation,
        top_k=TOP_K,
        random_seed=RANDOM_SEED,
    )

comparison = pd.DataFrame(summaries).T
comparison

## Inspect stability by date

A model should not be judged from one aggregate. These plots expose the distribution of daily rank IC and top-k market-relative return across validation dates.


In [ ]:
rank_ic_by_model = pd.concat(
    {name: result["rank_ic"] for name, result in daily_results.items()},
    axis=1,
)
rank_ic_by_model.plot.box(figsize=(10, 5), title="Validation daily rank IC")
plt.axhline(0, linewidth=1)
plt.ylabel("Spearman correlation")
plt.show()


In [ ]:
top_k_return_by_model = pd.concat(
    {name: result["top_k_mean_return"] for name, result in daily_results.items()},
    axis=1,
)
top_k_return_by_model.plot.box(
    figsize=(10, 5),
    title=f"Validation top-{TOP_K} market-relative return",
)
plt.axhline(0, linewidth=1)
plt.ylabel("Return")
plt.show()


## Inspect one validation date

Choose a date to compare the ranked candidates with the outcomes used only for evaluation.


In [ ]:
inspection_date = X_validation.index.get_level_values("trading_date").max()
inspection = pd.DataFrame(
    {name: scores for name, scores in validation_scores.items()}
).join(
    bundle.targets.loc[X_validation.index, [
        relative_return_column,
        percentile_column,
        relevance_column,
    ]]
)
inspection.xs(inspection_date, level="trading_date").sort_values(
    "ranking",
    ascending=False,
).head(TOP_K)


## Optional feature-importance inspection

Gain importance is a model diagnostic, not proof that a feature is stable or causal.


In [ ]:
feature_importance = pd.Series(
    ranker.feature_importances_,
    index=X_rank_train.columns,
).sort_values(ascending=False)
feature_importance.head(20).sort_values().plot.barh(
    figsize=(9, 7),
    title="XGBRanker feature importance",
    zorder=3,
)
plt.grid(zorder=1)
plt.xlabel("Importance")
plt.show()


In [ ]:
feature_importance = pd.Series(
    regressor.feature_importances_,
    index=X_train.columns,
).sort_values(ascending=False)
feature_importance.head(20).sort_values().plot.barh(
    figsize=(9, 7),
    title="XGBRegressor feature importance",
    zorder=3,
)
plt.grid(zorder=1)
plt.xlabel("Importance")
plt.show()

In [ ]:
feature_importance = pd.Series(
    classifier.feature_importances_,
    index=X_train.columns,
).sort_values(ascending=False)
feature_importance.sort_values().plot.barh(
    figsize=(9, 7),
    title="XGBClassifier feature importance",
    zorder=3,
)
plt.grid(zorder=1)
plt.xlabel("Importance")
plt.show()


In [ ]:
import seaborn as sns

importance_by_model = pd.DataFrame(
    {
        "XGBRanker": pd.Series(ranker.feature_importances_, index=X_rank_train.columns),
        "XGBRegressor": pd.Series(regressor.feature_importances_, index=X_train.columns),
        "XGBClassifier": pd.Series(classifier.feature_importances_, index=X_train.columns),
    }
)
importance_by_model = importance_by_model.loc[
    importance_by_model.mean(axis=1).sort_values(ascending=False).index
]

fig, ax = plt.subplots(figsize=(8, max(6, 0.35 * len(importance_by_model))))
sns.heatmap(
    importance_by_model,
    cmap="tab20b",
    vmin=0,
    vmax=0.1,
    annot=True,
    fmt=".3f",
    linewidths=0.5,
    cbar_kws={"label": "Importance"},
    ax=ax,
)
ax.set_title("Feature importance by model")
plt.tight_layout()
plt.show()


## Inspect predictions alongside validation candles


In [ ]:
columns=[
    "open",
    "high",
    "low",
    "close",
    "adjusted_close",
    "volume",
]

prices = load_bronze_daily_prices(
    engine=ENGINE,
    provider=PROVIDER,
    start_date=VALIDATION_START,
    end_date=VALIDATION_END,
    columns=columns,
    tickers=TICKERS,
).set_index(["provider", "ticker", "trading_date"])

prices

In [ ]:
def add_annotation(x, y, kind: str, value: float, fig: go.Figure) -> None:
    font_color = "#39da89" if kind == "low" else "#eb4343"
    fig.add_annotation(
        x=x,
        y=y,
        xref="x",
        yref="y",
        text=str(value),
        showarrow=True,
        font=dict(
            family="Courier New, monospace",
            size=10,
            color=font_color,
            weight=900,
        ),
        align="center",
        arrowhead=2,
        arrowsize=1,
        arrowwidth=2,
        arrowcolor="#636363",
        ax=0,
        ay=5 * np.log2(y) if kind == "low" else -5 * np.log2(y),
        bordercolor="#c7c7c7",
        borderwidth=1,
        borderpad=3,
        bgcolor="#484848",
        opacity=0.8,
        row=1, col=1,
    )

In [ ]:
iterator = iter(
    prices.sort_index(level=["ticker", "trading_date"], ascending=[True, True])
    .groupby("ticker", sort=False)
)

In [ ]:
ticker, frame = next(iterator)
pivots = indicators.pivot_points_high_low(
    data=frame,
    high_left=15,
    high_right=15,
    low_left=15,
    low_right=15,
    rank_output="strength",
    kind="balanced",
)
frame = pd.concat([frame, pivots], axis=1)


fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.01, row_heights=[0.6, 0.4],
)

# -----------------------
# SUBPLOT #1
# -----------------------

fig.add_candlestick(
    x=frame.index.get_level_values("trading_date"),
    open=frame["open"],
    high=frame["high"],
    low=frame["low"],
    close=frame["close"],
    line_width=1,
    row=1,
    col=1,
)

atr_14 = indicators.atr(frame, length=14)
stop_price = (frame["open"] - atr_14.mul(1).shift(1)).rename("stop_price")
tp_price = (frame["open"] + 2 * atr_14.mul(1).shift(1)).rename("tp_price")

extra_lines = pd.concat(
    [
        indicators.ema(frame["close"], length=10).rename("ema10"),
        indicators.ema(frame["close"], length=20).rename("ema20"),
        indicators.ema(frame["close"], length=50).rename("ema50"),
        indicators.ema(frame["close"], length=150).rename("ema150"),
        indicators.ema(frame["close"], length=200).rename("ema200"),
        stop_price,
        tp_price,
    ],
    axis=1,
)
for name, series in extra_lines.items():
    fig.add_scatter(
        x=series.index.get_level_values("trading_date"),
        y=series.values,
        line_shape="hvh",
        name=name,
        row=1,
        col=1,
    )

# Plot high/low annotations
highs = frame.query("pivot_high")["high"]
lows = frame.query("pivot_low")["low"]

for ind, y in lows.items():
    x = ind[-1]
    y = round(y, 1)
    add_annotation(x, y, kind="low", value=y, fig=fig)
for ind, y in highs.items():
    x = ind[-1]
    y = round(y, 1)
    add_annotation(x, y, kind="high", value=y, fig=fig)


# -----------------------
# SUBPLOT #2
# -----------------------

extra_lines = pd.concat(
    [
        df_scores.query("ticker == @ticker").drop(columns=["random"]),
        # features.donchian_position(frame, length=50),
    ],
    axis=1,
).astype(float)

for name, series in extra_lines.items():
    fig.add_scatter(
        x=series.index.get_level_values("trading_date"),
        y=series.values,
        name=name,
        row=2,
        col=1,
    )

fig.update_layout(
    xaxis_rangeslider_visible=False,
    title_text=ticker,
    height=750,
    hovermode="x unified",
    hoversubplots="axis",
)
fig.show()

In [ ]:

nrows = 1
ncols = df_scores.shape[1] - 1
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4), layout="constrained")
for ax, (mdl_name, series) in zip(axes, df_scores.drop(columns=["random"]).items(), strict=False):
    ax.hist(
        series,
        bins="auto",
        histtype="step",
        linewidth=2,
        cumulative=False,
        density=False,
        log=False,
        zorder=3,
    )
    ax.grid(zorder=1, which="both")
    ax.set_xlabel(mdl_name, fontweight="bold")
plt.show()

In [ ]:
temp = df_scores.drop(columns=["random"])
mean_score = temp.mean()
std_score = temp.std()
zscores = (temp - mean_score).div(std_score)

nrows = 1
ncols = zscores.shape[1]
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3), layout="constrained")
for ax, (mdl_name, series) in zip(axes, zscores.items(), strict=False):
    ax.hist(
        series,
        bins="auto",
        histtype="step",
        linewidth=2,
        cumulative=True,
        density=True,
        log=False,
        zorder=3,
    )
    ax.axvline(0, color="k", linewidth=1, zorder=2)
    ax.grid(zorder=1, which="both")
    ax.set_xlabel(f"Z({mdl_name})", fontweight="bold")
    ax.set_xlim(-3, 7)
    ax.set_xticks(np.arange(-3, 7.01, 1))
    ax.set_yticks(np.arange(0, 1.01, 0.1))
plt.show()

In [ ]:
from itertools import combinations

zscores_and_targets = pd.concat(
    [
        zscores,
        relative_return_validation,
        percentile_validation,
        relevance_validation,
    ],
    axis=1,
)

xy_cols = list(combinations(zscores.columns, 2))
zcols = zscores_and_targets.drop(columns=zscores.columns).columns
nrows = len(xy_cols)
ncols = len(zcols)
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 2.8 * nrows), layout="constrained")
for ax_row, (xcol, ycol) in zip(axes, xy_cols):
    for ax, zcol in zip(ax_row, zcols):
        if ax in axes[0]:
            ax.set_title(zcol, fontsize=13)
        ax.set_ylabel(ycol, fontweight="bold")
        ax.set_xlabel(xcol, fontweight="bold")
        xvals = zscores_and_targets[xcol]
        yvals = zscores_and_targets[ycol]
        zvals = zscores_and_targets[zcol]
        ax.scatter(
            x=xvals,
            y=yvals,
            c=zvals,
            s=1,
            alpha=0.3,
            cmap="Blues",
            zorder=3,
        )
        ax.grid(zorder=1)
        ax.set_xlim(
            xvals.quantile((0.001, 0.999))
        )
        ax.set_ylim(
            yvals.quantile((0.001, 0.999))
        )
plt.show()

## Save a study result

Export materially changed runs to HTML under `notebooks/exports/`, then restore this notebook to its generic base state before committing unrelated experiments.
